In [ ]:
def shift_features_2_candle(data):
    for i in ['open', 'close', 'low', 'high', 'volume', 'pattern']:
        data[f"{i}_N"] = data[i]
        data[f'{i}_N-1'] = data[i].shift(1)
    data.drop(['open', 'close', 'low', 'high', 'volume', 'pattern'], axis=1, inplace=True)
    data.dropna(inplace=True)
    for i in ['open', 'close', 'low', 'high', 'volume', 'pattern']:
        data[f'{i}_N-1'] = data[f'{i}_N-1'].astype('int32')
    return data


In [ ]:
def features_of_candle(data):
    # 1. Размер тела
    data['body_N'] = data['close_N'] - data['open_N']
    data['body_N-1'] = data['close_N-1'] - data['open_N']
    
    # 2. Размер всей свечи (high - low)
    data['candle_height_N'] = data['high_N'] - data['low_N']
    data['candle_height_N-1'] = data['high_N-1'] - data['low_N-1']
    
    # 3. Верхняя тень
    data['shadow_up_N'] = np.where(
        data['open_N'] < data['close_N'], 
        data['high_N'] - data['close_N'],
        data['high_N'] - data['open_N'])
    
    data['shadow_up_N-1'] = np.where(
        data['open_N-1'] < data['close_N-1'], 
        data['high_N-1'] - data['close_N-1'],
        data['high_N-1'] - data['open_N-1'])
    
    # 4. Нижняя тень
    data['shadow_down_N'] = np.where(
        data['open_N'] < data['close_N'],
        data['open_N'] - data['low_N'],
        data['close_N'] - data['low_N'])
    
    data['shadow_down_N-1'] = np.where(
        data['open_N-1'] < data['close_N-1'],
        data['open_N-1'] - data['low_N-1'],
        data['close_N-1'] - data['low_N-1'])
    
    # 5. Разница между телами (body_N - body_N-1)
    data['body_diff'] = data['body_N'] - data['body_N-1']
    
    # 6. Сумма теней
    data['shadow_sum_N'] = data['shadow_up_N'] + data['shadow_down_N']
    data['shadow_sum_N-1'] = data['shadow_up_N-1'] + data['shadow_down_N-1']
    
    # 7. Отношение верхней тени к телу (логически правильный метод с тремя случаями)
    data['ratio_up_shadow_to_body_N'] = np.where(
    data['body_N'] != 0,
    data['shadow_up_N'] / data['body_N'],
    np.where(
        data['shadow_up_N'] == 0,
        0,
        100))

    data['ratio_up_shadow_to_body_N_abs'] = np.where(
    data['body_N'] != 0,
    np.abs(data['shadow_up_N'] / data['body_N']),
    np.where(
        data['shadow_up_N'] == 0,
        0,
        100))

    data['ratio_up_shadow_to_body_N-1'] = np.where(
    data['body_N-1'] != 0,
    data['shadow_up_N-1'] / data['body_N-1'],
    np.where(
        data['shadow_up_N-1'] == 0,
        0,
        100))

    data['ratio_up_shadow_to_body_N-1_abs'] = np.where(
    data['body_N-1'] != 0,
    np.abs(data['shadow_up_N-1'] / data['body_N-1']),
    np.where(
        data['shadow_up_N-1'] == 0,
        0,
        100))


    # 8. Отношение нижней тени ко всему телу по модулу и просто
    data['ratio_down_shadow_to_body_N'] = np.where(
    data['body_N'] != 0,
    data['shadow_down_N'] / data['body_N'],
    np.where(
        data['shadow_down_N'] == 0,
        0,
        100))

    data['ratio_down_shadow_to_body_N_abs'] = np.where(
    data['body_N'] != 0,
    np.abs(data['shadow_down_N'] / data['body_N']),
    np.where(
        data['shadow_down_N'] == 0,
        0,
        100))

    data['ratio_down_shadow_to_body_N-1'] = np.where(
    data['body_N-1'] != 0,
    data['shadow_down_N-1'] / data['body_N-1'],
    np.where(
        data['shadow_down_N-1'] == 0,
        0,
        100))

    data['ratio_down_shadow_to_body_N-1_abs'] = np.where(
    data['body_N-1'] != 0,
    np.abs(data['shadow_down_N-1'] / data['body_N-1']),
    np.where(
        data['shadow_down_N-1'] == 0,
        0,
        100))
    
    # 9. Отношение суммы теней ко всему телу по модулу и просто
    data['ratio_sum_shadow_to_body_N'] = np.where(
    data['body_N'] != 0,
    data['shadow_sum_N'] / data['body_N'],
    np.where(
        data['shadow_sum_N'] == 0,
        0,
        100))

    data['ratio_sum_shadow_to_body_N_abs'] = np.where(
    data['body_N'] != 0,
    np.abs(data['shadow_sum_N'] / data['body_N']),
    np.where(
        data['shadow_sum_N'] == 0,
        0,
        100))

    data['ratio_sum_shadow_to_body_N-1'] = np.where(
    data['body_N-1'] != 0,
    data['shadow_sum_N-1'] / data['body_N-1'],
    np.where(
        data['shadow_sum_N-1'] == 0,
        0,
        100))

    data['ratio_sum_shadow_to_body_N-1_abs'] = np.where(
    data['body_N-1'] != 0,
    np.abs(data['shadow_sum_N-1'] / data['body_N-1']),
    np.where(
        data['shadow_sum_N-1'] == 0,
        0,
        100))
    
    # 10. Насколько % одна свеча перекрывает другую. Будем смотреть относительно тела N
    data['ratio_of_body'] = np.where(((data['body_N'] != 0) &  (data['body_N-1'] != 0)), 
                                     data['body_N'] / data['body_N-1'], 0)
                                     
    data['ratio_of_body_abs'] = np.where(((data['body_N'] != 0) &  (data['body_N-1'] != 0)), 
                                     np.abs(data['body_N'] / data['body_N-1']), 0)
    
    # 11. Флаг, если тело = 0 
    data['is_zero_body_N'] = (data['body_N'] == 0).astype(int)
    data['is_zero_body_N-1'] = (data['body_N-1'] == 0).astype(int)

    return data


In [ ]:
def detection_bullish_engulfing_pattern(data, filtr='NO'):
    """
    Функция для детекции бычьего паттерна 'поглощение' (Bullish Engulfing) на ценовых данных.
    Функция позволяет применять дополнительные фильтры для подтверждения надежности паттерна.

    Параметры:
    ----------
    data : pd.DataFrame
        DataFrame с ценовыми данными, должен содержать колонки:
        - 'open': цены открытия
        - 'close': цены закрытия
        - 'high', 'low', 'volume' для расширенных фильтров
        
    filtr : str, optional, default='NO'
        Тип фильтра для подтверждения паттерна:
        
        Базовые фильтры:
        - 'NO' - детекция паттерна без дополнительных фильтров
        
        Фильтры подтверждения бычьим движением:
        - '1_grow_candle_after_pattern' - паттерн подтверждается 1 растущей свечой после него
        - '2_grow_candle_after_pattern' - паттерн подтверждается 2 растущими свечами после него
        
        - '1_loss_candle_before_pattern' - паттерн подтверждается 1 падающей свечой до него  
        - '2_loss_candle_before_pattern' - паттерн подтверждается 2 падающими свечами до него
        
        - 'full_engulfing' - полное поглощение предыдущей свечи (вместе с тенями)
        

    Возвращает:
    -----------
    pd.DataFrame
        Исходный DataFrame с добавленной колонкой:
        - 'pattern': бинарный indicator (0/1), где 1 обозначает наличие паттерна"""
        
    # Сдивгаем данные:
    def shift_data_3(data):
        l = []
        for i in range(2, 0, -1):
            data_N_sh = data.shift(i)
            data_N_shi = data_N_sh.add_suffix(f'_N-{i}')
            l.append(data_N_shi)
        data_N = data.add_suffix('_N')
        l.append(data_N)
        for i in range(1, 4):
            data_N_sh = data.shift(-i)
            data_N_shi = data_N_sh.add_suffix(f'_N+{i}')
            l.append(data_N_shi)
        data_full = pd.concat(l, axis=1)
        data_full = data_full.dropna()
        return data_full
    
    # Получаем новый датафрейм
    data = shift_data_3(data)
    
    # Базовые переменные
    body = data['close_N'] - data['open_N']
    fut_body = data['close_N+1'] - data['open_N+1']
    open = data['open_N']
    close = data['close_N']
    fut_open = data['open_N+1']
    fut_close = data['close_N+1']
    
    # Переменные для фильтров
    fut_2_body = data['close_N+2'] - data['open_N+2']
    fut_3_body = data['close_N+3'] - data['open_N+3']
    prev_body = data['close_N-1'] - data['open_N-1']
    prev_2_body = data['close_N-2'] - data['open_N-2']
    high = data['high_N']
    low = data['low_N']

    condition = ((body < 0) &      # Базовое условие
                 (fut_body > 0) &          
                 (open < fut_close) & 
                 (close > fut_open))
    
    if filtr == 'NO':
        # Фильтра нет
        pattern_mask = condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == '1_grow_candle_after_pattern':
        # Фильтр: следующая после паттерна свеча должна быть растущей
        new_condition = condition & (fut_2_body > 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        
    elif filtr == '2_grow_candle_after_pattern':
        # Фильтр: 2 следующие после паттерна свечи должны быть растущими 
        new_condition = condition & (fut_2_body > 0) & (fut_3_body > 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == '1_loss_candle_before_pattern':
        # Фильтр: предыдущая свеча ДО паттерна должна быть падающей
        new_condition = condition & (prev_body < 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        
    elif filtr == '2_loss_candle_before_pattern':
        # Фильтр: 2 предыдущих свечи ДО паттерна должны быть падающими
        new_condition = condition & (prev_body < 0) & (prev_2_body < 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == 'full_engulfing':
        # Фильтр: следующая свечка перекрывает пердыдущую полностью(вместе с тенями)
        new_condition = (body < 0) & (fut_body > 0)  & (fut_close > high) & (fut_open < low)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        

In [ ]:
def detection_bearish_engulfing_pattern(data, filtr='NO'):
    """
    Функция для детекции медвежьего паттерна 'поглощение' (Bearish Engulfing) на ценовых данных.
    Функция позволяет применять дополнительные фильтры для подтверждения надежности паттерна.

    Параметры:
    ----------
    data : pd.DataFrame
        DataFrame с ценовыми данными, должен содержать колонки:
        - 'open': цены открытия
        - 'close': цены закрытия
        - 'high', 'low', 'volume' для расширенных фильтров
        
    filtr : str, optional, default='NO'
        Тип фильтра для подтверждения паттерна:
        
        Базовые фильтры:
        - 'NO' - детекция паттерна без дополнительных фильтров
        
        Фильтры подтверждения медвежьим движением:
        - '1_loss_candle_after_pattern' - паттерн подтверждается 1 падающей свечой после него
        - '2_loss_candle_after_pattern' - паттерн подтверждается 2 падающими свечами после него
        
        - '1_grow_candle_before_pattern' - паттерн подтверждается 1 растущей свечой до него  
        - '2_grow_candle_before_pattern' - паттерн подтверждается 2 растущими свечами до него
        
        - 'full_engulfing' - полное поглощение следующей свечей максимумом и минимумов предыдущей свечи
    Возвращает:
    -----------
    pd.DataFrame
        Исходный DataFrame с добавленной колонкой:
        - 'pattern': бинарный indicator (0/1), где 1 обозначает наличие паттерна
    """
    
       # Сдивгаем данные:
    def shift_data_3(data):
        l = []
        for i in range(2, 0, -1):
            data_N_sh = data.shift(i)
            data_N_shi = data_N_sh.add_suffix(f'_N-{i}')
            l.append(data_N_shi)
        data_N = data.add_suffix('_N')
        l.append(data_N)
        for i in range(1, 4):
            data_N_sh = data.shift(-i)
            data_N_shi = data_N_sh.add_suffix(f'_N+{i}')
            l.append(data_N_shi)
        data_full = pd.concat(l, axis=1)
        data_full = data_full.dropna()
        return data_full
    
    # Получаем новый датафрейм
    data = shift_data_3(data)
    
    # Базовые переменные
    body = data['close_N'] - data['open_N']
    fut_body = data['close_N+1'] - data['open_N+1']
    open = data['open_N']
    close = data['close_N']
    fut_open = data['open_N+1']
    fut_close = data['close_N+1']
    
    # Переменные для фильтров
    fut_2_body = data['close_N+2'] - data['open_N+2']
    fut_3_body = data['close_N+3'] - data['open_N+3']
    prev_body = data['close_N-1'] - data['open_N-1']
    prev_2_body = data['close_N-2'] - data['open_N-2']
    high = data['high_N']
    low = data['low_N']

    condition = ((body > 0) &      # Базовое условие
                 (fut_body < 0) &          
                 (open > fut_close) & 
                 (close < fut_open))
    
    if filtr == 'NO':
        # Фильтра нет
        pattern_mask = condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == '1_loss_candle_after_pattern':
        # Фильтр: следующая после паттерна свеча должна быть падающей
        new_condition = condition & (fut_2_body < 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        
    elif filtr == '2_loss_candle_after_pattern':
        # Фильтр: 2 следующие после паттерна свечи должны быть падающими
        new_condition = condition & (fut_2_body < 0) & (fut_3_body < 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == '1_grow_candle_before_pattern':
        # Фильтр: предыдущая свеча ДО паттерна должна быть растущей
        new_condition = condition & (prev_body > 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        
    elif filtr == '2_grow_candle_before_pattern':
        # Фильтр: 2 предыдущих свечи ДО паттерна должны быть растущими
        new_condition = condition & (prev_body > 0) & (prev_2_body > 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == 'full_engulfing':
        # Фильтр: следующая свечка перекрывает предыдущую полностью(вместе с тенями)
        new_condition = (body > 0) & (fut_body < 0)  & (fut_open > high) & (fut_close < low)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data

In [ ]:
def detection_bullish_harami(data, filtr='NO'):
        """
    Функция для детекции бычьего паттерна 'харами' (Bullish Harami) на ценовых данныx.
    Паттерн Харами состоит из большой медвежьей свечи, за которой следует маленькая бычья свеча,
    полностью содержащаяся в пределах тела предыдущей свечи.

    Параметры:
    ----------
    data : pd.DataFrame
        DataFrame с ценовыми данными, должен содержать колонки:
        - 'open': цены открытия
        - 'close': цены закрытия  
        - 'high', 'low', 'volume' для расширенных фильтров
        
    filtr : str, optional, default='NO'
        Тип фильтра для подтверждения паттерна:
        
        Базовые фильтры:
        - 'NO' - детекция паттерна без дополнительных фильтров
        
        Фильтры подтверждения бычьим движением:
        - '1_grow_candle_after_pattern' - паттерн подтверждается 1 растущей свечой после него
        - '2_grow_candle_after_pattern' - паттерн подтверждается 2 растущими свечами после него
        
        - '1_loss_candle_before_pattern' - паттерн подтверждается 1 падающей свечой до него  
        - '2_loss_candle_before_pattern' - паттерн подтверждается 2 падающими свечами до него
        
        - 'full_harami' - полное содержание следующей свечи в пределах тела предыдущей (включая тени)

    Возвращает:
    -----------
    pd.DataFrame
        Исходный DataFrame с добавленной колонкой:
        - 'pattern': бинарный indicator (0/1), где 1 обозначает наличие паттерна
    """
    
       # Сдивгаем данные:
        def shift_data_3(data):
            l = []
            for i in range(2, 0, -1):
                data_N_sh = data.shift(i)
                data_N_shi = data_N_sh.add_suffix(f'_N-{i}')
                l.append(data_N_shi)
            data_N = data.add_suffix('_N')
            l.append(data_N)
            for i in range(1, 4):
                data_N_sh = data.shift(-i)
                data_N_shi = data_N_sh.add_suffix(f'_N+{i}')
                l.append(data_N_shi)
            data_full = pd.concat(l, axis=1)
            data_full = data_full.dropna()
            return data_full
    
        # Получаем новый датафрейм
        data = shift_data_3(data)
    
        # Базовые переменные
        body = data['close_N'] - data['open_N']
        fut_body = data['close_N+1'] - data['open_N+1']
        open = data['open_N']
        close = data['close_N']
        fut_open = data['open_N+1']
        fut_close = data['close_N+1']
        
        # Переменные для фильтров
        fut_2_body = data['close_N+2'] - data['open_N+2']
        fut_3_body = data['close_N+3'] - data['open_N+3']
        fut_high = data['high_N+1']
        fut_low = data['low_N+1']
        prev_body = data['close_N-1'] - data['open_N-1']
        prev_2_body = data['close_N-2'] - data['open_N-2']


        condition = ((body < 0) &      # Базовое условие
                 (fut_body > 0) &          
                 (open > fut_close) & 
                 (close < fut_open))
    
        if filtr == 'NO':
            # Фильтра нет
            pattern_mask = condition
            data['pattern'] = pattern_mask.astype(int)
            return data
        
        elif filtr == '1_grow_candle_after_pattern':
            # Фильтр: следующая после паттерна свеча должна быть растущей
            new_condition = condition & (fut_2_body > 0)
            pattern_mask = new_condition
            data['pattern'] = pattern_mask.astype(int)
            return data
            
        elif filtr == '2_grow_candle_after_pattern':
            # Фильтр: 2 следующие после паттерна свечи должны быть растущими
            new_condition = condition & (fut_2_body > 0) & (fut_3_body > 0)
            pattern_mask = new_condition
            data['pattern'] = pattern_mask.astype(int)
            return data
        
        elif filtr == '1_loss_candle_before_pattern':
            # Фильтр: предыдущая свеча ДО паттерна должна быть падающей
            new_condition = condition & (prev_body < 0)
            pattern_mask = new_condition
            data['pattern'] = pattern_mask.astype(int)
            return data
            
        elif filtr == '2_loss_candle_before_pattern':
            # Фильтр: 2 предыдущих свечи ДО паттерна должны быть падающими
            new_condition = condition & (prev_body < 0) & (prev_2_body < 0)
            pattern_mask = new_condition
            data['pattern'] = pattern_mask.astype(int)
            return data
        
        elif filtr == 'full_harami':
            # Фильтр: Предыдущая свечка перекрывает будущую полностью(вместе с тенями)
            new_condition = (body < 0) & (fut_body > 0)  & (open > fut_high) & (close < fut_low)
            pattern_mask = new_condition
            data['pattern'] = pattern_mask.astype(int)
            return data

In [ ]:
def detection_bearish_harami(data, filtr='NO'):
    """
    Функция для детекции медвежьего паттерна 'харами' (Bearish Harami) на ценовых данных.
    Паттерн Харами состоит из большой бычьей свечи, за которой следует маленькая медвежья свеча,
    полностью содержащаяся в пределах тела предыдущей свечи. Является сигналом разворота вниз.

    Параметры:
    ----------
    data : pd.DataFrame
        DataFrame с ценовыми данными, должен содержать колонки:
        - 'open': цены открытия
        - 'close': цены закрытия  
        - 'high', 'low', 'volume' для расширенных фильтров
        
    filtr : str, optional, default='NO'
        Тип фильтра для подтверждения паттерна:
        
        Базовые фильтры:
        - 'NO' - детекция паттерна без дополнительных фильтров
        
        Фильтры подтверждения медвежьим движением:
        - '1_loss_candle_after_pattern' - паттерн подтверждается 1 падающей свечой после него
        - '2_loss_candle_after_pattern' - паттерн подтверждается 2 падающими свечами после него
        
        - '1_grow_candle_before_pattern' - паттерн подтверждается 1 растущей свечой до него  
        - '2_grow_candle_before_pattern' - паттерн подтверждается 2 растущими свечами до него
        
        - 'full_harami' - полное содержание следующей свечи в пределах тела предыдущей (включая тени)

    Возвращает:
    -----------
    pd.DataFrame
        Исходный DataFrame с добавленной колонкой:
        - 'pattern': бинарный indicator (0/1), где 1 обозначает наличие паттерна
    """
    
    # Сдивгаем данные:
    def shift_data_3(data):
        l = []
        for i in range(2, 0, -1):
            data_N_sh = data.shift(i)
            data_N_shi = data_N_sh.add_suffix(f'_N-{i}')
            l.append(data_N_shi)
        data_N = data.add_suffix('_N')
        l.append(data_N)
        for i in range(1, 4):
            data_N_sh = data.shift(-i)
            data_N_shi = data_N_sh.add_suffix(f'_N+{i}')
            l.append(data_N_shi)
        data_full = pd.concat(l, axis=1)
        data_full = data_full.dropna()
        return data_full
    
    # Получаем новый датафрейм
    data = shift_data_3(data)
    
    # Базовые переменные
    body = data['close_N'] - data['open_N']
    fut_body = data['close_N+1'] - data['open_N+1']
    open = data['open_N']
    close = data['close_N']
    fut_open = data['open_N+1']
    fut_close = data['close_N+1']
        
    # Переменные для фильтров
    fut_2_body = data['close_N+2'] - data['open_N+2']
    fut_3_body = data['close_N+3'] - data['open_N+3']
    fut_high = data['high_N+1']
    fut_low = data['low_N+1']
    prev_body = data['close_N-1'] - data['open_N-1']
    prev_2_body = data['close_N-2'] - data['open_N-2']


    condition = ((body > 0) &      # Базовое условие
                 (fut_body < 0) &          
                 (open < fut_close) & 
                 (close > fut_open))
    
    if filtr == 'NO':
        # Фильтра нет
        pattern_mask = condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        
    elif filtr == '1_loss_candle_after_pattern':
        # Фильтр: следующая после паттерна свеча должна быть падающей
        new_condition = condition & (fut_2_body < 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
            
    elif filtr == '2_loss_candle_after_pattern':
        # Фильтр: 2 следующие после паттерна свечи должны быть падающими
        new_condition = condition & (fut_2_body < 0) & (fut_3_body < 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == '1_grow_candle_before_pattern':
        # Фильтр: предыдущая свеча ДО паттерна должна быть растущей
        new_condition = condition & (prev_body > 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
        
    elif filtr == '2_grow_candle_before_pattern':
        # Фильтр: 2 предыдущих свечи ДО паттерна должны быть растущими
        new_condition = condition & (prev_body > 0) & (prev_2_body > 0)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data
    
    elif filtr == 'full_harami':
        # Фильтр: Предыдущая свечка перекрывает будущую полностью(вместе с тенями)
        new_condition = (body > 0) & (fut_body < 0)  & (open < fut_low) & (close > fut_high)
        pattern_mask = new_condition
        data['pattern'] = pattern_mask.astype(int)
        return data


In [ ]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd


# Альтернативная версия для работы с числовыми индексами
def visualize_patterns_simple(data, num_candles=100, start_from=0, pattern_col='pattern'):
    """
    Простая визуализация определенного количества свечей
    
    Parameters:
    -----------
    data : pd.DataFrame
        Данные с паттернами
    num_candles : int
        Количество свечей для отображения
    start_from : int
        Начальная позиция
    pattern_col : str
        Колонка с паттернами
    """
    
    # Выбираем диапазон данных
    end_idx = min(start_from + num_candles, len(data))
    subset = data.iloc[start_from:end_idx].copy()
    
    fig = go.Figure()
    
    # Свечной график
    fig.add_trace(go.Candlestick(x=subset.index,
                                open=subset['open'],
                                high=subset['high'],
                                low=subset['low'],
                                close=subset['close'],
                                name='Price'))
    
    # Подсветка паттернов
    pattern_points = subset[subset[pattern_col] == 1]
    if not pattern_points.empty:
        fig.add_trace(go.Scatter(x=pattern_points.index,
                                y=pattern_points['high'] * 1.01,
                                mode='markers',
                                marker=dict(size=12, color='red', symbol='star'),
                                name='Pattern'))
    
    # Сигналы
    if 'signal' in subset.columns:
        signal_points = subset[subset['signal'] == 1]
        if not signal_points.empty:
            fig.add_trace(go.Scatter(x=signal_points.index,
                                    y=signal_points['low'] * 0.99,
                                    mode='markers',
                                    marker=dict(size=12, color='blue', symbol='triangle-up'),
                                    name='Buy Signal'))
    
    fig.update_layout(title=f"Patterns (candles {start_from}-{end_idx})",
                     xaxis_title='Index',
                     yaxis_title='Price',
                     template='plotly_white',
                     height=500)
    
    fig.update_xaxes(rangeslider_visible=False)
    
    return fig
data_with_patterns = detection_bullish_engulfing_pattern(sber_1h, filtr='full_engulfing')
fig = visualize_patterns_simple(data_with_patterns, num_candles=400, start_from=1100)
fig.show()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

def simple_stop_take_optimize(data, signal_col='signal', price_col='close'):
    """
    Простой подбор стоп-лосса и тейк-профита с сравнением с Buy & Hold
    """
    # Параметры для перебора
    stop_levels = np.arange(0.005, 0.1, 0.005)  # 1%, 2%, 3%, 5%
    take_levels = np.arange(0.005, 0.1, 0.005)  # 2%, 3%, 5%, 8%
    
    best_profit = -1000
    best_stop = 0
    best_take = 0
    
    # Перебираем комбинации
    for stop in stop_levels:
        for take in take_levels:
            profit = test_strategy(data, stop, take, signal_col, price_col)
            if profit > best_profit:
                best_profit = profit
                best_stop = stop
                best_take = take
    
    # Тестируем лучшую стратегию
    strategy_returns = backtest_with_stop_take(data, best_stop, best_take, signal_col, price_col)
    buy_hold_returns = data[price_col].pct_change().fillna(0)
    
    # Строим график
    plt.figure(figsize=(12, 6))
    
    # Доходность стратегии
    strategy_cumulative = (1 + strategy_returns).cumprod()
    plt.plot(strategy_cumulative.index, strategy_cumulative.values, 
             label=f'Стратегия (SL:{best_stop:.1%}, TP:{best_take:.1%})', linewidth=2)
    
    # Buy & Hold
    bh_cumulative = (1 + buy_hold_returns).cumprod()
    plt.plot(bh_cumulative.index, bh_cumulative.values, 
             label='Buy & Hold', linewidth=2, alpha=0.7)
    
    plt.title(f'Сравнение стратегии\nДоходность: {best_profit:.1%} vs Buy&Hold: {bh_cumulative.iloc[-1]-1:.1%}')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
    
    print(f"Лучшие параметры: Стоп-лосс {best_stop:.1%}, Тейк-профит {best_take:.1%}")
    print(f"Доходность стратегии: {best_profit:.1%}")
    print(f"Доходность Buy & Hold: {bh_cumulative.iloc[-1]-1:.1%}")
    
    return best_stop, best_take, best_profit

def test_strategy(data, stop_loss, take_profit, signal_col, price_col):
    """Тестирует одну комбинацию параметров"""
    signals = data[data[signal_col] == 1].index
    total_return = 1.0
    
    for signal_idx in signals:
        entry_idx = data.index.get_loc(signal_idx)
        entry_price = data.iloc[entry_idx][price_col]
        
        # Ищем выход в следующие 5 дней
        for i in range(1, min(6, len(data) - entry_idx)):
            current_price = data.iloc[entry_idx + i][price_col]
            ret = (current_price - entry_price) / entry_price
            
            # Выход по стопу или тейку
            if ret <= -stop_loss or ret >= take_profit or i == 5:
                total_return *= (1 + ret)
                break
    
    return total_return - 1

def backtest_with_stop_take(data, stop_loss, take_profit, signal_col, price_col):
    """Бэктест с фиксированными параметрами"""
    returns = []
    dates = []
    
    signals = data[data[signal_col] == 1].index
    
    for signal_idx in signals:
        entry_idx = data.index.get_loc(signal_idx)
        entry_price = data.iloc[entry_idx][price_col]
        
        for i in range(1, min(6, len(data) - entry_idx)):
            current_idx = entry_idx + i
            current_price = data.iloc[current_idx][price_col]
            ret = (current_price - entry_price) / entry_price
            
            if ret <= -stop_loss or ret >= take_profit or i == 5:
                returns.append(ret)
                dates.append(data.index[current_idx])
                break
    
    return pd.Series(returns, index=dates)

# Использование
best_stop, best_take, profit = simple_stop_take_optimize(sber_1h_bull)